# COMPX525-26A Assignment 1 (V2): Petri Dish Cell Culture Classification

**Improved version with stronger models and advanced techniques.**

Three Approaches:
1. Custom CNN from scratch (deeper, with SE blocks)
2. Fine-tuning ConvNeXt-Tiny (stronger than EfficientNet-B0)
3. Pretrained embeddings + ML (DINOv2 features + SVM/XGBoost)

Key improvements over V1:
- Focal Loss instead of weighted CE
- Label smoothing
- Only use ORIGINAL images for validation (no augmented leakage)
- Stronger augmentation pipeline
- Progressive resizing (train at 224, fine-tune at 288)

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm albumentations scikit-learn xgboost

In [ ]:
import os, random, time, json, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T
import torchvision.models as models
import timm

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════
BASE_DIR = '/content/drive/MyDrive/deep_learning_1'  # <-- UPDATE THIS

TRAIN_DIR = os.path.join(BASE_DIR, 'processed', 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'processed', 'test')

IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 2
NUM_CLASSES  = 16  # a-i, k-q

for d in [TRAIN_DIR, TEST_DIR]:
    assert os.path.isdir(d), f'Not found: {d}'
print('Paths OK')

In [ ]:
# Build class mapping
all_classes = sorted([d for d in os.listdir(TRAIN_DIR)
                      if os.path.isdir(os.path.join(TRAIN_DIR, d)) and d != 'j'])
class_to_idx = {c: i for i, c in enumerate(all_classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
print(f'Classes ({len(all_classes)}): {all_classes}')

# Separate ORIGINAL vs AUGMENTED files by filename prefix
orig_files, orig_labels = [], []
aug_files, aug_labels = [], []
orig_class_counts = {}

for cls in all_classes:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    orig_count = 0
    for f in sorted(os.listdir(cls_dir)):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(cls_dir, f)
        if f.startswith('aug_'):
            aug_files.append(fpath)
            aug_labels.append(class_to_idx[cls])
        else:
            orig_files.append(fpath)
            orig_labels.append(class_to_idx[cls])
            orig_count += 1
    orig_class_counts[cls] = orig_count

orig_files = np.array(orig_files)
orig_labels = np.array(orig_labels)
aug_files = np.array(aug_files)
aug_labels = np.array(aug_labels)

print(f'\nOriginal: {len(orig_files)} images')
print(f'Augmented: {len(aug_files)} images')
for cls, cnt in sorted(orig_class_counts.items(), key=lambda x: x[1]):
    bar = '█' * (cnt // 20)
    print(f'  {cls}: {cnt:5d} {bar}')

In [ ]:
# Stratified split on ORIGINAL data only
train_idx, val_idx = train_test_split(
    np.arange(len(orig_files)), test_size=0.15,
    stratify=orig_labels, random_state=SEED
)

# Train = original train split + ALL augmented files
train_files  = np.concatenate([orig_files[train_idx], aug_files])
train_labels = np.concatenate([orig_labels[train_idx], aug_labels])

# Val = ONLY original images (no augmented data!)
val_files  = orig_files[val_idx]
val_labels = orig_labels[val_idx]

print(f'Clean split:')
print(f'  Train: {len(train_files)} ({len(orig_files[train_idx])} orig + {len(aug_files)} aug)')
print(f'  Val:   {len(val_files)} (original only, no augmented!)')

## 2. Datasets, Transforms & DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transforms(img_size=224):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.7, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.01, 0.05), p=1),
            A.GaussianBlur(blur_limit=(3, 5), p=1),
            A.MotionBlur(blur_limit=5, p=1),
        ], p=0.3),
        A.OneOf([
            A.OpticalDistortion(distort_limit=0.05, p=1),
            A.GridDistortion(num_steps=5, distort_limit=0.05, p=1),
            A.ElasticTransform(alpha=1, sigma=50, p=1),
        ], p=0.2),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
        A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, img_size//16), hole_width_range=(1, img_size//16), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=224):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size=224):
    return [
        get_val_transforms(img_size),
        A.Compose([A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                   A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size), A.VerticalFlip(p=1.0),
                   A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size), A.RandomRotate90(p=1.0),
                   A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
    ]

print('Transforms ready')

In [ ]:
class PetriDataset(Dataset):
    def __init__(self, file_list, labels, transform=None):
        self.files = file_list
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.files[idx]).convert('RGB'))
        label = self.labels[idx]
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


class TestPetriDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.files = sorted([f for f in os.listdir(test_dir)
                             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = np.array(Image.open(os.path.join(self.test_dir, fname)).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, fname

In [ ]:
# Create datasets and loaders
train_dataset = PetriDataset(train_files, train_labels, get_train_transforms(IMG_SIZE))
val_dataset   = PetriDataset(val_files, val_labels, get_val_transforms(IMG_SIZE))

# Weighted sampler (lighter weights - sqrt inverse frequency)
train_counts = Counter(train_labels)
# Use sqrt of inverse frequency instead of pure inverse - less aggressive
sample_weights = np.array([1.0 / np.sqrt(train_counts[l]) for l in train_labels])
sample_weights = torch.from_numpy(sample_weights).double()
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} samples, {len(val_loader)} batches')

## 3. Loss Functions & Training Utilities

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss - focuses on hard-to-classify examples."""
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.alpha = alpha  # class weights tensor

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets, weight=self.alpha,
            reduction='none', label_smoothing=self.label_smoothing
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


# Compute balanced class weights (lighter than pure inverse)
total = sum(train_counts.values())
class_weights = torch.tensor(
    [np.sqrt(total / (NUM_CLASSES * train_counts[i])) for i in range(NUM_CLASSES)],
    dtype=torch.float32
).to(DEVICE)
# Normalize so mean = 1
class_weights = class_weights / class_weights.mean()
print('Class weights (sqrt-balanced):')
for i, w in enumerate(class_weights):
    print(f'  {idx_to_class[i]}: {w:.3f}')

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler=None, scaler=None):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    if scheduler is not None:
        scheduler.step()

    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        if criterion:
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

        probs = F.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    acc = 100.0 * correct / total
    loss = running_loss / total if criterion else 0
    probs = np.concatenate(all_probs, axis=0)
    return loss, acc, np.array(all_preds), np.array(all_labels), probs


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs, model_name, patience=12, use_amp=True):
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    no_improve = 0
    save_path = os.path.join(BASE_DIR, f'best_{model_name}_v2.pth')
    scaler = GradScaler() if use_amp else None

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler
        )
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        lr = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0
        print(f'Epoch [{epoch+1}/{num_epochs}] '
              f'Loss: {train_loss:.4f}/{val_loss:.4f} '
              f'Acc: {train_acc:.1f}%/{val_acc:.1f}% '
              f'LR: {lr:.6f} ({elapsed:.0f}s)')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f'  >> Best model saved ({val_acc:.2f}%)')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'\nBest: {best_val_acc:.2f}%')
    return model, history

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title(f'{title} - Loss')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title(f'{title} - Accuracy')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, f'v2_{title.lower().replace(" ", "_")}_curves.png'), dpi=150)
    plt.show()

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=all_classes, yticklabels=all_classes, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix - {title}')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, f'v2_{title.lower().replace(" ", "_")}_cm.png'), dpi=150)
    plt.show()

---
## 4. Approach 1: Custom CNN from Scratch (with SE Attention)

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation attention block."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        b, c, _, _ = x.size()
        w = F.adaptive_avg_pool2d(x, 1).view(b, c)
        w = F.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w)).view(b, c, 1, 1)
        return x * w


class SEResBlock(nn.Module):
    """Residual block with SE attention."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return F.relu(out + self.shortcut(x))


class CustomSENet(nn.Module):
    """Custom CNN with SE-Residual blocks."""
    def __init__(self, num_classes=16):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1))

        self.layer1 = self._make_layer(64,  64,  3, stride=1)
        self.layer2 = self._make_layer(64,  128, 4, stride=2)
        self.layer3 = self._make_layer(128, 256, 6, stride=2)
        self.layer4 = self._make_layer(256, 512, 3, stride=2)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes))

        self._init_weights()

    def _make_layer(self, in_ch, out_ch, n_blocks, stride):
        layers = [SEResBlock(in_ch, out_ch, stride)]
        for _ in range(1, n_blocks):
            layers.append(SEResBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.head(x)


custom_model = CustomSENet(NUM_CLASSES).to(DEVICE)
params = sum(p.numel() for p in custom_model.parameters())
print(f'Custom SE-Net: {params:,} parameters')

In [ ]:
# Train Custom SE-Net
EPOCHS_1 = 60
criterion_1 = FocalLoss(alpha=class_weights, gamma=2.0, label_smoothing=0.1)
optimizer_1 = optim.AdamW(custom_model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler_1 = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_1, T_0=15, T_mult=2, eta_min=1e-6)

print('Training Custom SE-Net...')
print('=' * 70)
custom_model, hist_custom = train_model(
    custom_model, train_loader, val_loader, criterion_1,
    optimizer_1, scheduler_1, EPOCHS_1,
    model_name='custom_senet', patience=20
)

In [ ]:
plot_history(hist_custom, 'Custom SE-Net')
_, acc_custom, preds_c, labels_c, probs_c = evaluate(custom_model, val_loader, criterion_1)
print(f'Custom SE-Net Val Accuracy: {acc_custom:.2f}%')
print(classification_report(labels_c, preds_c, target_names=all_classes))
plot_cm(labels_c, preds_c, 'Custom SE-Net')

---
## 5. Approach 2: Fine-tuning ConvNeXt-Tiny

ConvNeXt is a modern pure-CNN architecture that matches Vision Transformers.

In [ ]:
def create_convnext(num_classes, freeze_ratio=0.7):
    """Create ConvNeXt-Tiny with partial freezing."""
    model = timm.create_model('convnext_tiny.fb_in22k_ft_in1k', pretrained=True, num_classes=num_classes)

    # Freeze early layers (first 70% of parameters)
    all_params = list(model.parameters())
    freeze_until = int(len(all_params) * freeze_ratio)
    for i, p in enumerate(all_params):
        if i < freeze_until:
            p.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'ConvNeXt-Tiny: {total:,} total, {trainable:,} trainable ({100*trainable/total:.0f}%)')
    return model

convnext = create_convnext(NUM_CLASSES, freeze_ratio=0.7).to(DEVICE)

In [ ]:
# Phase 1: Train with partial freeze
EPOCHS_2A = 15
criterion_2 = FocalLoss(alpha=class_weights, gamma=2.0, label_smoothing=0.1)
optimizer_2a = optim.AdamW(
    filter(lambda p: p.requires_grad, convnext.parameters()),
    lr=5e-4, weight_decay=1e-4
)
scheduler_2a = optim.lr_scheduler.CosineAnnealingLR(optimizer_2a, T_max=EPOCHS_2A, eta_min=1e-6)

print('Phase 1: Partial fine-tune (30% trainable)...')
print('=' * 70)
convnext, hist_2a = train_model(
    convnext, train_loader, val_loader, criterion_2,
    optimizer_2a, scheduler_2a, EPOCHS_2A,
    model_name='convnext_phase1', patience=10
)

In [ ]:
# Phase 2: Unfreeze all, discriminative LR
EPOCHS_2B = 35
for p in convnext.parameters():
    p.requires_grad = True

# Layer-wise learning rates
param_groups = []
all_named = list(convnext.named_parameters())
n_layers = len(all_named)
base_lr = 2e-5
for i, (name, param) in enumerate(all_named):
    # Earlier layers get lower LR (linear scaling)
    layer_lr = base_lr * (0.1 + 0.9 * (i / n_layers))
    param_groups.append({'params': param, 'lr': layer_lr})

optimizer_2b = optim.AdamW(param_groups, weight_decay=1e-4)
scheduler_2b = optim.lr_scheduler.CosineAnnealingLR(optimizer_2b, T_max=EPOCHS_2B, eta_min=1e-7)

trainable = sum(p.numel() for p in convnext.parameters() if p.requires_grad)
print(f'Phase 2: Full fine-tune ({trainable:,} params, layer-wise LR)')
print('=' * 70)
convnext, hist_2b = train_model(
    convnext, train_loader, val_loader, criterion_2,
    optimizer_2b, scheduler_2b, EPOCHS_2B,
    model_name='convnext_finetuned', patience=15
)

In [ ]:
hist_convnext = {k: hist_2a[k] + hist_2b[k] for k in hist_2a}
plot_history(hist_convnext, 'ConvNeXt-Tiny')
_, acc_convnext, preds_cn, labels_cn, probs_cn = evaluate(convnext, val_loader, criterion_2)
print(f'ConvNeXt-Tiny Val Accuracy: {acc_convnext:.2f}%')
print(classification_report(labels_cn, preds_cn, target_names=all_classes))
plot_cm(labels_cn, preds_cn, 'ConvNeXt-Tiny')

---
## 6. Approach 3: Pretrained Embeddings + ML

In [ ]:
@torch.no_grad()
def extract_features(model, loader, is_test=False):
    model.eval()
    feats, labs = [], []
    for batch in loader:
        images = batch[0].to(DEVICE)
        f = model(images)
        feats.append(f.cpu().numpy())
        if not is_test:
            labs.append(batch[1].numpy())
    feats = np.concatenate(feats)
    if not is_test:
        return feats, np.concatenate(labs)
    return feats

# Use ConvNeXt-Base as feature extractor (stronger than ResNet-50)
feat_model = timm.create_model('convnext_base.fb_in22k_ft_in1k', pretrained=True, num_classes=0)
feat_model = feat_model.to(DEVICE).eval()

# Create loaders with val_transform for consistent features
embed_train_ds = PetriDataset(orig_files[train_idx], orig_labels[train_idx], get_val_transforms(IMG_SIZE))
embed_val_ds   = PetriDataset(val_files, val_labels, get_val_transforms(IMG_SIZE))
embed_train_loader = DataLoader(embed_train_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
embed_val_loader   = DataLoader(embed_val_ds,   batch_size=64, shuffle=False, num_workers=NUM_WORKERS)

print('Extracting features...')
X_train, y_train = extract_features(feat_model, embed_train_loader)
X_val, y_val     = extract_features(feat_model, embed_val_loader)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

scaler_feat = StandardScaler()
X_train_sc = scaler_feat.fit_transform(X_train)
X_val_sc   = scaler_feat.transform(X_val)

# Free GPU memory
del feat_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# SVM
print('Training SVM...')
svm = SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced',
          probability=True, random_state=SEED)
svm.fit(X_train_sc, y_train)
svm_preds = svm.predict(X_val_sc)
svm_acc = accuracy_score(y_val, svm_preds) * 100
print(f'SVM: {svm_acc:.2f}%')

# XGBoost
print('\nTraining XGBoost...')
xgb = XGBClassifier(
    n_estimators=1000, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=SEED, n_jobs=-1, tree_method='hist',
    early_stopping_rounds=50
)
xgb.fit(X_train_sc, y_train, eval_set=[(X_val_sc, y_val)], verbose=100)
xgb_preds = xgb.predict(X_val_sc)
xgb_acc = accuracy_score(y_val, xgb_preds) * 100
print(f'XGBoost: {xgb_acc:.2f}%')

# Random Forest
print('\nTraining Random Forest...')
rf = RandomForestClassifier(n_estimators=1000, class_weight='balanced',
                            random_state=SEED, n_jobs=-1)
rf.fit(X_train_sc, y_train)
rf_preds = rf.predict(X_val_sc)
rf_acc = accuracy_score(y_val, rf_preds) * 100
print(f'Random Forest: {rf_acc:.2f}%')

# Best ML
ml_results = {'svm': (svm_acc, svm), 'xgb': (xgb_acc, xgb), 'rf': (rf_acc, rf)}
best_ml_name = max(ml_results, key=lambda k: ml_results[k][0])
best_ml_acc, best_ml_clf = ml_results[best_ml_name]
print(f'\nBest ML: {best_ml_name} ({best_ml_acc:.2f}%)')

best_ml_preds = best_ml_clf.predict(X_val_sc)
print(classification_report(y_val, best_ml_preds, target_names=all_classes))
plot_cm(y_val, best_ml_preds, f'Embeddings + {best_ml_name.upper()}')

---
## 7. Ensemble & Predictions

In [ ]:
# ── Validation Ensemble ──
_, _, _, _, val_probs_custom  = evaluate(custom_model, val_loader)
_, _, _, _, val_probs_convnext = evaluate(convnext, val_loader)

val_probs_ml = best_ml_clf.predict_proba(X_val_sc) if hasattr(best_ml_clf, 'predict_proba') \
    else np.eye(NUM_CLASSES)[best_ml_clf.predict(X_val_sc)]

# Find optimal weights via grid search on validation
best_w, best_ens_acc = None, 0
for w1 in np.arange(0.05, 0.4, 0.05):
    for w2 in np.arange(0.2, 0.7, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        ens = w1 * val_probs_custom + w2 * val_probs_convnext + w3 * val_probs_ml
        acc = accuracy_score(val_labels, ens.argmax(1)) * 100
        if acc > best_ens_acc:
            best_ens_acc = acc
            best_w = (w1, w2, w3)

w_custom, w_convnext, w_ml = best_w
print(f'Optimal weights: Custom={w_custom:.2f}, ConvNeXt={w_convnext:.2f}, ML={w_ml:.2f}')

val_ens_probs = w_custom * val_probs_custom + w_convnext * val_probs_convnext + w_ml * val_probs_ml
val_ens_preds = val_ens_probs.argmax(1)
ens_acc = accuracy_score(val_labels, val_ens_preds) * 100

print(f'\n--- RESULTS ---')
print(f'Custom SE-Net:  {acc_custom:.2f}%')
print(f'ConvNeXt-Tiny:  {acc_convnext:.2f}%')
print(f'Best ML ({best_ml_name}):  {best_ml_acc:.2f}%')
print(f'Ensemble:       {ens_acc:.2f}%')

print('\nEnsemble Report:')
print(classification_report(val_labels, val_ens_preds, target_names=all_classes))
plot_cm(val_labels, val_ens_preds, 'Ensemble')

In [ ]:
# ── Test Predictions with TTA ──
@torch.no_grad()
def predict_with_tta(model, test_dir, tta_list):
    model.eval()
    all_probs = []
    for t in tta_list:
        ds = TestPetriDataset(test_dir, t)
        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
        probs = []
        for images, _ in loader:
            images = images.to(DEVICE)
            out = model(images)
            probs.append(F.softmax(out, dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs))
    return np.mean(all_probs, axis=0)

print('Generating test predictions with TTA...')
tta_list = get_tta_transforms(IMG_SIZE)

test_probs_custom  = predict_with_tta(custom_model, TEST_DIR, tta_list)
test_probs_convnext = predict_with_tta(convnext, TEST_DIR, tta_list)

# ML test features
feat_model2 = timm.create_model('convnext_base.fb_in22k_ft_in1k', pretrained=True, num_classes=0)
feat_model2 = feat_model2.to(DEVICE).eval()
test_embed_loader = DataLoader(
    TestPetriDataset(TEST_DIR, get_val_transforms(IMG_SIZE)),
    batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
X_test = extract_features(feat_model2, test_embed_loader, is_test=True)
X_test_sc = scaler_feat.transform(X_test)
del feat_model2; gc.collect(); torch.cuda.empty_cache()

test_probs_ml = best_ml_clf.predict_proba(X_test_sc) if hasattr(best_ml_clf, 'predict_proba') \
    else np.eye(NUM_CLASSES)[best_ml_clf.predict(X_test_sc)]

# Ensemble
test_probs = w_custom * test_probs_custom + w_convnext * test_probs_convnext + w_ml * test_probs_ml
test_preds = test_probs.argmax(1)

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
test_classes = [idx_to_class[p] for p in test_preds]

print(f'Predictions: {len(test_files)} images')
print(f'Distribution: {dict(Counter(test_classes))}')

In [ ]:
# Save CSV
csv_path = os.path.join(BASE_DIR, 'submission_v2.csv')
with open(csv_path, 'w') as f:
    f.write('TestFileName, Class\n')
    for fname, cls in zip(test_files, test_classes):
        f.write(f'{fname}, {cls}\n')
print(f'Saved: {csv_path}')

# Show first rows
df = pd.read_csv(csv_path)
print(df.head(10).to_string(index=False))

In [ ]:
# ── Save all results to JSON ──
results_v2 = {
    'custom_senet': {
        'val_accuracy': round(acc_custom, 2),
        'history': {k: [round(v, 4) for v in vals] for k, vals in hist_custom.items()},
        'report': classification_report(labels_c, preds_c, target_names=all_classes, output_dict=True),
    },
    'convnext_tiny': {
        'val_accuracy': round(acc_convnext, 2),
        'history': {k: [round(v, 4) for v in vals] for k, vals in hist_convnext.items()},
        'report': classification_report(labels_cn, preds_cn, target_names=all_classes, output_dict=True),
    },
    'embeddings_ml': {
        'svm': round(svm_acc, 2),
        'xgboost': round(xgb_acc, 2),
        'random_forest': round(rf_acc, 2),
        'best': best_ml_name,
        'best_accuracy': round(best_ml_acc, 2),
        'report': classification_report(y_val, best_ml_preds, target_names=all_classes, output_dict=True),
    },
    'ensemble': {
        'val_accuracy': round(ens_acc, 2),
        'weights': {'custom': w_custom, 'convnext': w_convnext, 'ml': w_ml},
        'report': classification_report(val_labels, val_ens_preds, target_names=all_classes, output_dict=True),
    },
    'test': {
        'total': len(test_files),
        'distribution': dict(Counter(test_classes)),
    }
}

with open(os.path.join(BASE_DIR, 'results_v2.json'), 'w') as f:
    json.dump(results_v2, f, indent=2, default=str)
print('Results saved to results_v2.json')

print(f'\n{"="*50}')
print(f'FINAL SUMMARY')
print(f'{"="*50}')
print(f'Custom SE-Net:  {acc_custom:.2f}%')
print(f'ConvNeXt-Tiny:  {acc_convnext:.2f}%')
print(f'Best ML ({best_ml_name}):  {best_ml_acc:.2f}%')
print(f'ENSEMBLE:       {ens_acc:.2f}%')
print(f'{"="*50}')

## 8. Model Comparison

In [ ]:
model_names = ['Custom SE-Net', 'ConvNeXt-Tiny\n(Fine-tuned)', f'Embeddings +\n{best_ml_name.upper()}', 'Ensemble']
accuracies = [acc_custom, acc_convnext, best_ml_acc, ens_acc]
colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(model_names, accuracies, color=colors_bar, edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{acc:.2f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('V2 Model Comparison')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'v2_model_comparison.png'), dpi=150)
plt.show()